In [2]:
from datasets import load_dataset, load_from_disk
import os

def print_tree(root: str, prefix: str = "") -> None:
    entries = sorted(os.listdir(root))
    for i, name in enumerate(entries):
        path = os.path.join(root, name)
        is_last = (i == len(entries) - 1)
        branch = "└── " if is_last else "├── "
        print(prefix + branch + name)
        if os.path.isdir(path):
            extension = "    " if is_last else "│   "
            print_tree(path, prefix + extension)
            

print("Import Complete.")

Import Complete.


In [3]:
# Save to disk in a folder “/home/youruser/hub_data/librispeech”
root_dir = "hub_data/librispeech"
clean_dir = os.path.join(root_dir, "clean")
other_dir = os.path.join(root_dir, "other")

os.makedirs(clean_dir, exist_ok=True)
os.makedirs(other_dir, exist_ok=True)

print("Setup Complete.")

Setup Complete.


In [4]:
print("Downloading LibriSpeech ASR 'clean' configuration (train.100, train.360, validation, test)...")
ds_clean = load_dataset("openslr/librispeech_asr", "clean")

print("Downloading LibriSpeech ASR 'other' configuration (train.500, validation, test)...")
ds_other = load_dataset("openslr/librispeech_asr", "other")

print(f"Saving 'clean' to: {clean_dir}")
ds_clean.save_to_disk(clean_dir)

print(f"Saving 'other' to: {other_dir}")
ds_other.save_to_disk(other_dir)

print("\nSaved datasets. Split keys:")
print("clean splits:", list(ds_clean.keys()))
print("other splits:", list(ds_other.keys()))

print("\nExample split lengths (number of rows):")
for k in ds_clean.keys():
    print(f"  clean/{k}: {len(ds_clean[k])}")
for k in ds_other.keys():
    print(f"  other/{k}: {len(ds_other[k])}")

print("\nReloading from disk to verify...")
ds_clean_re = load_from_disk(clean_dir)
ds_other_re = load_from_disk(other_dir)

print("Reload OK. Reloaded split keys:")
print("clean:", list(ds_clean_re.keys()))
print("other:", list(ds_other_re.keys()))

print("\nFolder structure under ./hub_data:")
root_to_print = "./hub_data"
print(root_to_print)
print_tree(root_to_print)


Saving 'clean' to: hub_data/librispeech/clean


Saving the dataset (1/1 shards): 100%|██████████| 2703/2703 [00:00<00:00, 4233.42 examples/s]


Saving 'other' to: hub_data/librispeech/other


Saving the dataset (1/1 shards): 100%|██████████| 2864/2864 [00:01<00:00, 2542.18 examples/s]



Saved datasets. Split keys:
clean splits: ['test', 'train.100', 'train.360', 'validation']
other splits: ['test', 'train.500', 'validation']

Example split lengths (number of rows):
  clean/test: 2620
  clean/train.100: 28539
  clean/train.360: 104014
  clean/validation: 2703
  other/test: 2939
  other/train.500: 148688
  other/validation: 2864

Reloading from disk to verify...
Reload OK. Reloaded split keys:
clean: ['test', 'train.100', 'train.360', 'validation']
other: ['test', 'train.500', 'validation']

Folder structure under ./hub_data:
./hub_data
└── librispeech
    ├── clean
    │   ├── dataset_dict.json
    │   ├── test
    │   │   ├── data-00000-of-00001.arrow
    │   │   ├── dataset_info.json
    │   │   └── state.json
    │   ├── train.100
    │   │   ├── data-00000-of-00014.arrow
    │   │   ├── data-00001-of-00014.arrow
    │   │   ├── data-00002-of-00014.arrow
    │   │   ├── data-00003-of-00014.arrow
    │   │   ├── data-00004-of-00014.arrow
    │   │   ├── data-00005-o

In [2]:
"""
Download and cache the LibriSpeech LM dataset from HuggingFace.
Run this once before training the ELM.

Usage:
    python download_librispeech_lm.py
"""

import os
from datasets import load_dataset, config

print("=" * 60)
print("Downloading openslr/librispeech_lm from HuggingFace")
print("~1.5GB download, ~4.4GB on disk. May take 10-30 min.")
print("=" * 60)
print()

ds = load_dataset("openslr/librispeech_lm", split="train", trust_remote_code=True)

print(f"\nDataset loaded: {len(ds):,} rows")
print(f"Features: {list(ds.features.keys())}")
print(f"\nSample rows:")
for i in [0, 1, 2]:
    text = ds[i]['text']
    preview = text[:100] + "..." if len(text) > 100 else text
    print(f"  [{i}] {preview}")

# Show cache location
cache_dir = config.HF_DATASETS_CACHE
print(f"\nHuggingFace cache directory: {cache_dir}")

# Find the actual cached directory for this dataset
print(f"\nSearching for librispeech_lm in cache...")
lm_root = None
for root, dirs, files in os.walk(cache_dir):
    basename = os.path.basename(root).lower()
    if 'librispeech_lm' in basename or ('openslr' in root.lower() and 'librispeech' in root.lower()):
        if any(f.endswith('.arrow') for f in files) or any(f == 'dataset_info.json' for f in files):
            # Walk up to find the dataset root
            candidate = root
            while candidate != cache_dir:
                parent = os.path.dirname(candidate)
                parent_base = os.path.basename(parent).lower()
                if 'librispeech' in parent_base or 'openslr' in parent_base:
                    candidate = parent
                else:
                    break
            if lm_root is None or len(candidate) < len(lm_root):
                lm_root = candidate

if lm_root is None:
    # Fallback: search more broadly
    for entry in os.listdir(cache_dir):
        if 'librispeech' in entry.lower() or 'openslr' in entry.lower():
            lm_root = os.path.join(cache_dir, entry)
            break


def print_tree(path, prefix="", max_depth=4, current_depth=0):
    """Print a directory tree similar to the `tree` command."""
    if current_depth > max_depth:
        return

    entries = sorted(os.listdir(path))
    dirs = [e for e in entries if os.path.isdir(os.path.join(path, e))]
    files = [e for e in entries if os.path.isfile(os.path.join(path, e))]

    # Group arrow files for compact display
    arrow_files = [f for f in files if f.endswith('.arrow')]
    other_files = [f for f in files if not f.endswith('.arrow')]

    all_items = []

    for f in other_files:
        fpath = os.path.join(path, f)
        size = os.path.getsize(fpath)
        if size > 1024 * 1024 * 1024:
            size_str = f"({size / (1024**3):.2f} GB)"
        elif size > 1024 * 1024:
            size_str = f"({size / (1024**2):.1f} MB)"
        elif size > 1024:
            size_str = f"({size / 1024:.1f} KB)"
        else:
            size_str = f"({size} B)"
        all_items.append(('file', f, size_str))

    if arrow_files:
        if len(arrow_files) <= 4:
            for f in arrow_files:
                fpath = os.path.join(path, f)
                size = os.path.getsize(fpath)
                size_str = f"({size / (1024**2):.1f} MB)" if size > 1024*1024 else f"({size / 1024:.1f} KB)"
                all_items.append(('file', f, size_str))
        else:
            total_size = sum(os.path.getsize(os.path.join(path, f)) for f in arrow_files)
            if total_size > 1024 * 1024 * 1024:
                total_str = f"{total_size / (1024**3):.2f} GB"
            else:
                total_str = f"{total_size / (1024**2):.1f} MB"
            all_items.append(('file', arrow_files[0], ''))
            all_items.append(('arrow_summary', f"... {len(arrow_files)} arrow files total, {total_str}", ''))
            all_items.append(('file', arrow_files[-1], ''))

    for d in dirs:
        all_items.append(('dir', d, ''))

    for i, (kind, name, extra) in enumerate(all_items):
        is_last = (i == len(all_items) - 1)
        connector = "└── " if is_last else "├── "
        extension = "    " if is_last else "│   "

        if kind == 'arrow_summary':
            print(f"{prefix}{connector}{name}")
        elif kind == 'file':
            line = f"{prefix}{connector}{name}"
            if extra:
                line += f"  {extra}"
            print(line)
        else:
            print(f"{prefix}{connector}{name}/")
            print_tree(os.path.join(path, name), prefix + extension,
                       max_depth, current_depth + 1)


if lm_root and os.path.exists(lm_root):
    print(f"\nFolder structure under HF cache:")
    print(f"{lm_root}")
    print_tree(lm_root)

    # Total size
    total = 0
    for root, dirs, files in os.walk(lm_root):
        for f in files:
            total += os.path.getsize(os.path.join(root, f))
    print(f"\nTotal disk usage: {total / (1024**3):.2f} GB")
else:
    print("\n  Could not auto-detect cache folder layout.")
    print(f"  Files are cached under: {cache_dir}")
    print(f"  Run: find {cache_dir} -name '*librispeech*' -type d")

print(f"\n{'=' * 60}")
print("Dataset is now cached. Future loads will be instant.")
print("The CharMambaLM training script uses this automatically.")
print(f"{'=' * 60}") 

~1.5GB download, ~4.4GB on disk. May take 10-30 min.




Dataset loaded: 40,418,260 rows
Features: ['text']

Sample rows:
  [0] A
  [1] A A
  [2] A A A

HuggingFace cache directory: /home/roberh18/.cache/huggingface/datasets

Searching for librispeech_lm in cache...

Folder structure under HF cache:
/home/roberh18/.cache/huggingface/datasets/openslr___librispeech_asr/other/0.0.0/71cacbfb7e2354c4226d01e70d77d5fca3d04ba1
├── dataset_info.json  (11.4 KB)
├── librispeech_asr-test.arrow
├── ... 65 arrow files total, 30.29 GB
└── librispeech_asr-validation.arrow

Total disk usage: 30.29 GB

Dataset is now cached. Future loads will be instant.
The CharMambaLM training script uses this automatically.
